# GRU Networks

Check a scalar gated update, inspect PyTorch's GRU output shapes, and fit a tiny order task. The task is a mechanics demonstration; it is not a benchmark.

In [1]:
import torch
from torch import nn

torch.manual_seed(24)
torch.set_num_threads(1)
dtype = torch.float64
print('CPU | GRU gate arithmetic and sequence checks')

CPU | GRU gate arithmetic and sequence checks


## Verify the update interpolation

Using the note's convention, a quarter update keeps three quarters of the old state and writes one quarter of the candidate.

In [2]:
previous = torch.tensor(0.6, dtype=dtype)
candidate = torch.tensor(-0.2, dtype=dtype)
update = torch.tensor(0.25, dtype=dtype)
current = (1 - update) * previous + update * candidate
torch.testing.assert_close(current, torch.tensor(0.4, dtype=dtype))
print(f'old={previous.item():.2f}, candidate={candidate.item():.2f}, update={update.item():.2f} -> new={current.item():.2f}')

old=0.60, candidate=-0.20, update=0.25 -> new=0.40


## Inspect a built-in GRU

The output contains a hidden vector for every time step. The final hidden state is returned separately and equals the last output for this single-direction, unpadded example.

In [3]:
layer = nn.GRU(input_size=2, hidden_size=4, batch_first=True).double()
sequence = torch.tensor([[[1., 0.], [0., 1.], [0., 0.]]], dtype=dtype)
outputs, final_state = layer(sequence)
assert outputs.shape == (1, 3, 4)
assert final_state.shape == (1, 1, 4)
torch.testing.assert_close(outputs[:, -1], final_state[-1])
print('outputs:', tuple(outputs.shape), '| final state:', tuple(final_state.shape))

outputs: (1, 3, 4) | final state: (1, 1, 4)


## Fit an order classifier

Two sequences contain the same marker and distractor, but their order determines the label. A GRU classifier reads the last output and predicts the class.

In [4]:
features, labels = [], []
for index in range(200):
    row = torch.zeros(6, 2)
    marker, distractor = (1, 4) if index % 2 == 0 else (4, 1)
    row[marker, 0] = 1
    row[distractor, 1] = 1
    features.append(row)
    labels.append(index % 2)
features = torch.stack(features)
labels = torch.tensor(labels)

class GRUClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.GRU(2, 8, batch_first=True)
        self.head = nn.Linear(8, 2)
    def forward(self, x):
        outputs, _ = self.rnn(x)
        return self.head(outputs[:, -1])

model = GRUClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
for _ in range(80):
    optimizer.zero_grad()
    loss = nn.functional.cross_entropy(model(features), labels)
    loss.backward()
    optimizer.step()
with torch.no_grad():
    accuracy = (model(features).argmax(1) == labels).float().mean().item()
assert accuracy > 0.95
print(f'train loss={loss.item():.4f} | toy accuracy={accuracy:.1%}')

train loss=0.0001 | toy accuracy=100.0%


This dataset reuses two templates and the reported training accuracy only confirms that the model can fit the tiny example. It says nothing about generalization. Hold out templates or data sources for a real evaluation, and account for padding when batching different sequence lengths.